NicheNet training: differentiation study example
================

In this tutorial, we will demonstrate the procedure for the scenario
where you want to explore cell-cell communication differences between
cell types, instead of between two conditions of the same cell type.
This corresponds to the “Cell Localization” scenario in the flowchart
below:

<img src="./images/flowchart.svg" style="width:70.0%" />

### Load data and networks

We will be using the mouse liver scRNA-seq data generated in the
[Guilliams et al (2022)
paper](https://www.sciencedirect.com/science/article/pii/S0092867421014811).
A subset of this data will be used for this tutorial. The full dataset can
be accessed at the [Liver Cell Atlas](https://livercellatlas.org/).

We will look at cell-cell communication differences between Kupffer
cells, the resident liver macrophages, and bile duct and capsule
macrophages. This means that we are interested in identifying the
Kupffer cell-specific ligands important for its identity. We will be
focusing on the niche cells of KCs, which are liver sinusoidal
endothelial cells (LSECs), hepatocytes, and stellate cells. Therefore,
we will skip the sender-agnostic approach for this analysis.

In [1]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import (
    LigandReceptorNetwork,
    WeightedNetwork
)
from nichenetpy.utils import (
    combine_by_key,
    combine_dicts
)
from nichenetpy.extraction import (
    get_expressed_genes,
    subset_ann,
    get_weighted_ligand_receptor_links,
    get_lfc_celltype
)
from nichenetpy.gene_symbol import mouse_alias_info
from nichenetpy.visualization import (
    prepare_ligand_target_visualization,
    prepare_ligand_receptor_visualization,
    heatmap_2d,
    heatmap_1d
)
from nichenetpy.metrics import group_metrics
from nichenetpy.io import read_ligand_target_matrix

from itertools import cycle, chain

import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import os
import requests
import pickle
import session_info

Download the model files

In [2]:
model_path = os.path.normpath("./tutorial_files/model/mouse")
if not os.path.exists(model_path):
    os.makedirs(model_path)
filename = "nichenet_mouse.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Download the AnnData object

In [3]:
data_path = os.path.normpath("./tutorial_files/AnnData")
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "seurat_obj_subset_integrated_zonation.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/15344079/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

In [4]:
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]
lr_network = model["lr_network"]
lr_sig = model["lr_sig"]

In [5]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "seurat_obj_subset_integrated_zonation.h5"))
mouse_alias_info.alias_to_symbol(ann)

KeyError: 'gene'

In [ ]:
ann

AnnData object with n_obs × n_vars = 5027 × 13541
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nGene', 'nUMI', 'aggregate', 'res.0.6', 'celltype'
    var: 'gene'
    layers: 'counts', 'data', 'scale.data'

In [ ]:
ann.obs

,orig.ident,nCount_RNA,nFeature_RNA,nGene,nUMI,aggregate,res.0.6,celltype
W380370,LN_SS,1607.0,876,880.0,1611.0,SS,1,CD8 T
W380372,LN_SS,885.0,536,541.0,891.0,SS,0,CD4 T
W380374,LN_SS,1223.0,737,742.0,1229.0,SS,0,CD4 T
W380378,LN_SS,1537.0,838,847.0,1546.0,SS,1,CD8 T
W380379,LN_SS,1603.0,836,839.0,1606.0,SS,0,CD4 T
...,...,...,...,...,...,...,...,...
W673547,LN_LCMV,1080.0,524,525.0,1081.0,LCMV,0,CD4 T
W673548,LN_LCMV,1002.0,613,615.0,1007.0,LCMV,0,CD4 T
W673549,LN_LCMV,7175.0,2122,2129.0,7182.0,LCMV,1,CD8 T
W673550,LN_LCMV,1348.0,660,661.0,1351.0,LCMV,0,CD4 T


Define a “receiver” cell population. The receiver cell population can only consist of one cell type.

In [ ]:
receiver = "KCs"

Determine which genes are expressed in the receiver cell population. The function get_expressed_genes considers genes to be expressed if they have non-zero counts in a certain percentage of the cell population (by default set at 10%). Users are also free to define expressed genes differently in a way that fits their data.

In [ ]:
expressed_genes_receiver = set(get_expressed_genes(receiver, ann, 0.1))

IndexError: list index out of range

Get a list of all receptors available in the ligand-receptor network, and define expressed receptors as genes that are in the ligand-receptor network and expressed in the receiver.

In [ ]:
all_receptors = lr_network.get_receptors()
expressed_receptors = all_receptors.intersection(expressed_genes_receiver)
potential_ligands = set(
    key for key, group in lr_network.item_iter()
    if len(group.intersection(expressed_receptors)) > 0
)

Define the gene set of interest that represents the cell-cell communication event to be studied. Perform DE analysis between the cell type of interest (KCs) and other localizations of the cell type (bile duct and capsule macrophages). Similar to above, we only retain genes that are significantly upregulated in KCs compared to both conditions.

In [ ]:
group_metrics(
    ann,
    group_oi=receiver,
    group_ref="MoMac1",
    groupby="aggregate",
    layer="data",
    min_pct=0.1,
    min_abs_lfc=0.25
)
DE_MoMac1 = ann.uns["group_metrics"]

In [ ]:
session_info()